In [1]:
include("../../src/NAML.jl")
using Oscar
using .NAML

  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.6.0


In [7]:
function pseudorandom_padic_int(p, prec, exp)
    K = PadicField(p, prec)
    coeffs = rand(0:(p-1), exp+1)
    return K(sum([coeffs[i] * p^(i-1) for i in Base.eachindex(coeffs)]))
end

function generate_random_function(vec::Vector)::Vector{Float64}
    return rand(0:1, length(vec))    
end

generate_random_function (generic function with 1 method)

In [11]:
# Initialise the p-adic field
prec = 20
p = 2
K = PadicField(p, prec)
# poly_degree = 7

function generate_data(n_leafs::Int)
    p_adic_p = O(K, 2)
    x = [p_adic_p ^ (-2) * pseudorandom_padic_int(p, prec, prec) for i in 1:n_leafs]
    y = generate_random_function(x)
    return (x, y)
end

generate_data (generic function with 1 method)

In [ ]:
# Take data points (x, y) and output the new training data inputs (x, x ^ 2, ...)
function polynomial_fitting_data(xy::Tuple, degree)
    (x, y) = xy
    return map(i -> x ^ i, 0:degree), y
end

function mk_cutoff(cutoff_val)::Function
    return x -> x < cutoff_val ? 0 : 1
end

# We shouldn't need this - this should be done automatically when we specialize!
function mk_linear_poly_loss(xy_vec::Vector{Tuple}, cutoff_val::Float64)
    @assert length(xy_vec) != 0 "Empty training data"
    linear_polynomials = Vector{NAML.PolydiscFunction{PadicFieldElem}}()
    cutoff_fun = mk_cutoff(cutoff_val)
    for (x, y) in xy_vec 
        # The linear function is (a_0, ..., a_n) -> a_0 + a_1 x + ... + a_n x ^ n - y
        linear = NAML.LinearAbsolutePolynomialSum([NAML.LinearPolynomial(x, K(0))])
        linear_transformed = (NAML.Comp(cutoff_fun, linear) - NAML.Constant{PadicFieldElem}(y))^ 2
        push!(linear_polynomials, linear_transformed)
    end
    main_function = NAML.batch_evaluate_init(sum(linear_polynomials))
    # main_function = NAML.batch_evaluate_init(NAML.Constant{PadicFieldElem}(56))
    batch_main_fn = input_arr::Vector{NAML.ValuationPolydisc{PadicFieldElem, Int64}} -> map(main_function, input_arr)
    return NAML.Loss(batch_main_fn, x -> 0)
end

function eval_function(x,)

# xy_vec::Vector{Tuple} = [([K(1), K(2)], 1)]

# fun = mk_linear_poly_loss(xy_vec, 1/8)


# @show fun.eval([NAML.ValuationPolydisc{PadicFieldElem, Int64}([K(0), K(1)], [100, 100])])

function mk_initial_for(n::Int)
    return NAML.ValuationPolydisc(zeros(K, n), zeros(Int64, n))
end

mk_initial_for (generic function with 1 method)

In [13]:
n_leafs = 3
degree_model = 6
x_vec, y_vec = generate_data(n_leafs)
xy_vec = Vector{Tuple}()
for i in 1:length(x_vec)
    xy = polynomial_fitting_data((x_vec[i], y_vec[i]), degree_model)
    push!(xy_vec, xy)
end

loss = mk_linear_poly_loss(xy_vec, 1/4)


Loss(var"#mk_linear_poly_loss##8#mk_linear_poly_loss##9"{Main.NAML.var"#batch_evaluate_init##5#batch_evaluate_init##6"{Main.NAML.var"#batch_evaluate_init##7#batch_evaluate_init##8"{Main.NAML.var"#batch_evaluate_init##13#batch_evaluate_init##14"{Main.NAML.SMul{PadicFieldElem}, Main.NAML.var"#batch_evaluate_init##9#batch_evaluate_init##10"{Main.NAML.var"#eval#batch_evaluate_init##16"{PadicFieldElem, Main.NAML.Constant{PadicFieldElem}}, Main.NAML.var"#eval#batch_evaluate_init##15"{PadicFieldElem, Main.NAML.Comp{PadicFieldElem}, Main.NAML.var"#eval#batch_evaluate_init##17"{PadicFieldElem, Vector{Function}}}}}, Main.NAML.var"#batch_evaluate_init##9#batch_evaluate_init##10"{Main.NAML.var"#eval#batch_evaluate_init##16"{PadicFieldElem, Main.NAML.Constant{PadicFieldElem}}, Main.NAML.var"#eval#batch_evaluate_init##15"{PadicFieldElem, Main.NAML.Comp{PadicFieldElem}, Main.NAML.var"#eval#batch_evaluate_init##17"{PadicFieldElem, Vector{Function}}}}}, Main.NAML.var"#batch_evaluate_init##5#batch_evalu

In [14]:
println("Initialising optimisation tools")
state = 1
# Here we want to force all coordinates to shrink little by little
config = (true, 0) #(false, 1) #
# Let's pick the Gauss point as our initial parameter
gauss_point = NAML.ValuationPolydisc([K(1) for i in 1:degree_model+1], zeros(Int64, degree_model+1))
# crazy_point = NAML.ValuationPolydisc([K(0) for i in 1:degree_model+1], [-6 for i in 0:degree_model])
# We optimise using Greedy descent
greedy_optim = NAML.greedy_descent_init(gauss_point, loss, state, config)
@show loss.eval([gauss_point])
# @show loss.eval([crazy_point])

Initialising optimisation tools
loss.eval([gauss_point]) = [2.0]


1-element Vector{Float64}:
 2.0

In [ ]:
N_epochs = 20
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch $i is $(NAML.eval_loss(greedy_optim))")
    println("----- Parameter is $(greedy_optim.param)")
    step!(greedy_optim)
end 
t2 = time()

Loss at epoch 1 is 2.0
----- Parameter is Polydisc over Field of 2-adic numbers with center PadicFieldElem[2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20)] and radius [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Loss at epoch 2 is 2.0
----- Parameter is Polydisc over Field of 2-adic numbers with center PadicFieldElem[2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20)] and radius [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Loss at epoch 3 is 2.0
----- Parameter is Polydisc over Field of 2-adic numbers with center PadicFieldElem[2^0 + O(2^20), 2^1 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20)] and radius [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Loss at epoch 4 is 2.0
----- Parameter is Polydis

1.76918053781024e9